# Exploring `loop.py` — The Dual While Loop Engine

This notebook is a hands-on walkthrough of `liteagent.loop` — the core of our agent library.  
We'll call the loop directly, watch events stream, see tool execution, and understand each moving part.

**What we'll cover:**
1. Why does `convert_to_llm` exist? (internal format vs what litellm accepts)
2. Simplest possible call — one LLM round, no tools
3. Adding a tool — echo
4. Error handling — tools that throw
5. Pydantic validation + type coercion
6. Multi-turn context — how messages accumulate
7. The `on_update` callback — streaming from tools
8. Multiple tool calls
9. Usage tracking
10. `agent_loop_continue` — resuming from manually-built context
11. Testing with different models
12. Steering — interrupting the loop mid-run
13. Follow-ups — the outer loop
14. Cancellation — `signal`
15. `transform_context` — modifying messages before each LLM call
16. `reasoning_effort` — thinking/reasoning

---

## Setup

The loop has **two entry points** (like pi's `agentLoop` and `agentLoopContinue`):

- `agent_loop(prompts, context, config, signal)` — start a new run with prompt messages
- `agent_loop_continue(context, config, signal)` — resume from existing context

Both return an `EventStream` immediately. The actual LLM call runs as an async task.

To use them, we need:
- **`AgentContext`** — system prompt + messages + tools (the *data* the loop works on)
- **`AgentConfig`** — model + convert_to_llm + hooks (the *behavior* of the loop)

In [1]:
from liteagent import (
    agent_loop,
    agent_loop_continue,
    AgentContext,
    AgentConfig,
    Tool,
    ToolResult,
)
import json

ALL_MODELS = [
    "anthropic/claude-sonnet-4-6",
    "anthropic/claude-opus-4-6",
    "gemini/gemini-3-flash-preview",
    "gemini/gemini-3.1-pro-preview",
    "gpt-5.2",
]

## 1. Why does `convert_to_llm` exist?

litellm accepts OpenAI-format messages and translates them for each provider. So why do
we need a `convert_to_llm` hook at all? Can't we just send our messages straight through?

Let's find out by looking at what the loop *actually stores* on messages — and what would
break if we sent that raw to litellm.

In [2]:
# What does an assistant message look like INSIDE the loop?
# (This is what context.messages contains after a run)
#
# Note: messages are plain dicts, not typed dataclasses.
# We have types for Tool, ToolResult, AgentContext, AgentConfig —
# but messages stay as dicts at the loop layer. Same as pi's agent-loop.ts.
# Typed message classes would live in the Agent class / app layer above.

example_assistant_msg = {
    "role": "assistant",
    "content": "Hello!",
    "tool_calls": None,
    "thinking_blocks": None,  # Anthropic cryptographic signatures
    "reasoning_content": None,  # universal thinking text
    "provider_specific_fields": None,  # Gemini thought_signatures, etc.
    "usage": {  # token counts — OUR addition
        "prompt_tokens": 15,
        "completion_tokens": 3,
        "total_tokens": 18,
        "cache_read_tokens": 0,
        "cache_creation_tokens": 0,
    },
    "stop_reason": "stop",  # OUR addition
    "timestamp": 1709312400000,  # OUR addition
}

example_tool_msg = {
    "role": "tool",
    "tool_call_id": "call_abc123",
    "name": "echo",
    "content": [{"type": "text", "text": "hello"}],  # list of content blocks
    "details": {"extra": "ui-only data"},  # OUR addition — not for the LLM
    "is_error": False,  # OUR addition
    "timestamp": 1709312401000,  # OUR addition
}

print("=== What the loop stores (our internal format) ===")
print(f"Assistant keys: {list(example_assistant_msg.keys())}")
print(f"Tool keys:      {list(example_tool_msg.keys())}")

print("\n=== What litellm/providers actually accept ===")
print("Assistant: role, content, tool_calls (+ thinking_blocks/reasoning_content)")
print("Tool:      role, tool_call_id, content (string or content blocks)")
print()
print("Extra keys that would confuse providers:")
print("  assistant: usage, stop_reason, timestamp, provider_specific_fields")
print("  tool:      details, is_error, timestamp, name")

=== What the loop stores (our internal format) ===
Assistant keys: ['role', 'content', 'tool_calls', 'thinking_blocks', 'reasoning_content', 'provider_specific_fields', 'usage', 'stop_reason', 'timestamp']
Tool keys:      ['role', 'tool_call_id', 'name', 'content', 'details', 'is_error', 'timestamp']

=== What litellm/providers actually accept ===
Assistant: role, content, tool_calls (+ thinking_blocks/reasoning_content)
Tool:      role, tool_call_id, content (string or content blocks)

Extra keys that would confuse providers:
  assistant: usage, stop_reason, timestamp, provider_specific_fields
  tool:      details, is_error, timestamp, name


### So `convert_to_llm` exists for three reasons:

**Reason 1: Strip our extras.** The loop enriches messages with metadata the LLM can't see
(`usage`, `stop_reason`, `timestamp`, `details`, `is_error`). These need to be stripped
before sending to the provider.

**Reason 2: App-level custom messages.** A real agent app (like pi's coding agent) might store
custom message types in the conversation — `bashExecution`, `branchSummary`, etc. The LLM
doesn't know what those are. `convert_to_llm` maps them to standard `user`/`assistant`/`tool`
messages.

Pi's default is literally a one-liner filter (converted to python)
```python
def default_convert_to_llm(messages):
    return [m for m in messages if m["role"] in ("user", "assistant", "tool")]
```

The coding agent overrides it with a real implementation that converts `bashExecution` →
`user` message with command + output, `branchSummary` → `user` message with summary text, etc.

**Reason 3 (provider quirks):** OpenAI requires tool results as plain strings, not content
block arrays. Anthropic/Gemini accept content blocks. `convert_to_llm` is where you handle
that — the loop doesn't need to know which provider you're using.

Now let's write the simplest possible one and use it:

In [3]:
# simple_convert: the minimal convert_to_llm
# - assistant: keep role/content/tool_calls/thinking, drop usage/stop_reason/timestamp
# - user: pass through
# - tool: keep role/tool_call_id/content (flatten to string), drop details/is_error/timestamp
def simple_convert(messages):
    result = []
    for m in messages:
        role = m.get("role")
        if role == "assistant":
            msg = {"role": "assistant"}
            if m.get("content"):
                msg["content"] = m["content"]
            if m.get("tool_calls"):
                msg["tool_calls"] = m["tool_calls"]
            if m.get("thinking_blocks"):
                msg["thinking_blocks"] = m["thinking_blocks"]
            if m.get("reasoning_content"):
                msg["reasoning_content"] = m["reasoning_content"]
            result.append(msg)
        elif role == "user":
            result.append({"role": "user", "content": m["content"]})
        elif role == "tool":
            content = m.get("content")
            if isinstance(content, list):
                text_parts = [b["text"] for b in content if b.get("type") == "text"]
                content = "\n".join(text_parts)
            result.append(
                {"role": "tool", "tool_call_id": m["tool_call_id"], "content": content}
            )
    return result


# Demo: what convert does to our enriched messages
raw = [example_assistant_msg, example_tool_msg]
clean = simple_convert(raw)

print("Before convert_to_llm:")
for m in raw:
    print(f"  [{m['role']}] keys={list(m.keys())}")

print("\nAfter convert_to_llm:")
for m in clean:
    print(f"  [{m['role']}] keys={list(m.keys())}")

Before convert_to_llm:
  [assistant] keys=['role', 'content', 'tool_calls', 'thinking_blocks', 'reasoning_content', 'provider_specific_fields', 'usage', 'stop_reason', 'timestamp']
  [tool] keys=['role', 'tool_call_id', 'name', 'content', 'details', 'is_error', 'timestamp']

After convert_to_llm:
  [assistant] keys=['role', 'content']
  [tool] keys=['role', 'tool_call_id', 'content']


## 2. Simplest possible call — one LLM round, no tools

Now that we know why `convert_to_llm` exists, let's use it. The absolute minimum:
send a prompt, get a response, watch the events stream back.

In [4]:
# MODEL — change this to test different providers
MODEL = "anthropic/claude-sonnet-4-6"

context = AgentContext(
    system_prompt="You are a helpful assistant. Be concise.",
    messages=[],
    tools=None,
)

config = AgentConfig(
    model=MODEL,
    convert_to_llm=simple_convert,
)

# agent_loop takes a list of messages to inject — any role works,
# but in practice these are user messages (the new prompt).
prompt_messages = [{"role": "user", "content": "What is 2+2? Answer in several words."}]

stream = agent_loop(prompt_messages, context, config)

# Consume the stream — every event printed
events = []
async for event in stream:
    print(event)

{'type': 'agent_start'}
{'type': 'turn_start'}
{'type': 'message_start', 'message': {'role': 'user', 'content': 'What is 2+2? Answer in several words.'}}
{'type': 'message_end', 'message': {'role': 'user', 'content': 'What is 2+2? Answer in several words.'}}
{'type': 'message_start', 'message': {'role': 'assistant', 'content': None, 'tool_calls': None}}
{'type': 'message_update', 'message': {'role': 'assistant', 'content': 'The', 'tool_calls': None}, 'delta': {'content': 'The'}, 'delta_type': 'text_delta'}
{'type': 'message_update', 'message': {'role': 'assistant', 'content': 'The answer is simply', 'tool_calls': None}, 'delta': {'content': ' answer is simply'}, 'delta_type': 'text_delta'}
{'type': 'message_update', 'message': {'role': 'assistant', 'content': 'The answer is simply **4**.', 'tool_calls': None}, 'delta': {'content': ' **4**.'}, 'delta_type': 'text_delta'}
{'type': 'message_end', 'message': {'role': 'assistant', 'content': 'The answer is simply **4**.', 'tool_calls': None

### What just happened — event sequence

For a simple no-tools call, the event sequence is:

```
agent_start          — loop begins
turn_start           — first LLM call
message_start        — user prompt echoed
message_end          — user prompt done
message_start        — assistant starts streaming
message_update (x N) — text deltas arrive token by token
message_end          — assistant message finalized
turn_end             — turn complete (no tool results)
agent_end            — loop done, messages returned
```

The stream's `.result()` gives you all new messages from this run.

In [5]:
# The final result — all **new** messages from this run
result = await stream.result()
result

[{'role': 'user', 'content': 'What is 2+2? Answer in several words.'},
 {'role': 'assistant',
  'content': 'The answer is simply **4**.',
  'tool_calls': None,
  'thinking_blocks': None,
  'reasoning_content': None,
  'provider_specific_fields': None,
  'usage': {'prompt_tokens': 30,
   'completion_tokens': 10,
   'total_tokens': 40,
   'cache_read_tokens': 0,
   'cache_creation_tokens': 0},
  'stop_reason': 'stop',
  'timestamp': 1772756073958}]

## 3. Adding a tool — echo

Tools are `Tool` dataclasses with:
- `name`, `description`, `parameters` — what the LLM sees (JSON Schema)
- `execute` — what we call: `async def(tool_call_id, params, signal, on_update) -> ToolResult`
- `params_model` (optional) — Pydantic BaseModel for validation + type coercion

When the LLM decides to call a tool, the loop:
1. Parses the JSON arguments
2. Validates with Pydantic (if `params_model` set)
3. Calls `execute()`
4. Wraps the result as a tool message
5. Sends it back to the LLM for the next turn

In [6]:
# Define the echo tool
async def echo_execute(tool_call_id, params, signal=None, on_update=None):
    return ToolResult(content=[{"type": "text", "text": params["message"]}])


echo_tool = Tool(
    name="echo",
    description="Echo back the given message. Use this when asked to echo something.",
    parameters={
        "type": "object",
        "properties": {
            "message": {"type": "string", "description": "The message to echo back"}
        },
        "required": ["message"],
    },
    execute=echo_execute,
)

print(f"Tool defined: {echo_tool.name}")
print(f"Parameters schema: {json.dumps(echo_tool.parameters, indent=2)}")

Tool defined: echo
Parameters schema: {
  "type": "object",
  "properties": {
    "message": {
      "type": "string",
      "description": "The message to echo back"
    }
  },
  "required": [
    "message"
  ]
}


In [7]:
context = AgentContext(
    system_prompt="You are helpful. When asked to echo, use the echo tool.",
    messages=[],
    tools=[echo_tool],
)
config = AgentConfig(model=MODEL, convert_to_llm=simple_convert)

prompt = [{"role": "user", "content": "Echo the message: hello world"}]
stream = agent_loop(prompt, context, config)

events = []
async for event in stream:
    print(event)

{'type': 'agent_start'}
{'type': 'turn_start'}
{'type': 'message_start', 'message': {'role': 'user', 'content': 'Echo the message: hello world'}}
{'type': 'message_end', 'message': {'role': 'user', 'content': 'Echo the message: hello world'}}
{'type': 'message_start', 'message': {'role': 'assistant', 'content': None, 'tool_calls': None}}
{'type': 'message_update', 'message': {'role': 'assistant', 'content': None, 'tool_calls': [{'id': 'toolu_01P3VHWLrTNGVKADhmvoUhG4', 'type': 'function', 'function': {'name': 'echo', 'arguments': ''}}]}, 'delta': {'tool_calls': [{'index': 0, 'id': 'toolu_01P3VHWLrTNGVKADhmvoUhG4', 'function': {'name': 'echo'}}]}, 'delta_type': 'tool_call_delta'}
{'type': 'message_update', 'message': {'role': 'assistant', 'content': None, 'tool_calls': [{'id': 'toolu_01P3VHWLrTNGVKADhmvoUhG4', 'type': 'function', 'function': {'name': 'echo', 'arguments': ''}}]}, 'delta': {'tool_calls': [{'index': 0}]}, 'delta_type': 'tool_call_delta'}
{'type': 'message_update', 'message'

### Tool call event sequence

With a tool call, the loop does two turns:

```
agent_start
turn_start                     ← turn 1: LLM decides to call a tool
  message_start (user)         ← prompt injected
  message_end (user)
  message_start (assistant)    ← empty skeleton {content: None, tool_calls: None}
  message_update (tool_call_delta, xN)  ← args stream in: {"messa → ge": "h → ello world"}
  message_end (assistant)      ← finalized with tool_calls filled in
  tool_execution_start         ← loop calls our execute() function
  tool_execution_end           ← tool returns ToolResult
  message_start (tool)         ← tool result wrapped as a message
  message_end (tool)
turn_end                       ← {message: assistant_msg, tool_results: [tool_msg]}
turn_start                     ← turn 2: LLM sees tool result, responds with text
  message_start (assistant)
  message_update (text_delta, xN)
  message_end (assistant)      ← stop_reason="stop"
turn_end                       ← {message: assistant_msg, tool_results: []}
agent_end                      ← {messages: [user, assistant, tool, assistant]}
```

**Key things to notice:**
- `message_update` deltas during tool calls are for UI preview only — the loop waits
  for `message_end` (built by `stream_chunk_builder`) before executing anything
- `turn_end` bundles the assistant message + its tool results — one event, full picture of the turn
- `agent_end` has ALL messages from the entire run across all turns
- The loop checks `tool_calls` presence (not `stop_reason`) to decide whether to continue

In [8]:
# Look at the messages that were returned
result = await stream.result()
result

[{'role': 'user', 'content': 'Echo the message: hello world'},
 {'role': 'assistant',
  'content': None,
  'tool_calls': [{'id': 'toolu_01P3VHWLrTNGVKADhmvoUhG4',
    'type': 'function',
    'function': {'name': 'echo', 'arguments': '{"message": "hello world"}'},
    'provider_specific_fields': None}],
  'thinking_blocks': None,
  'reasoning_content': None,
  'provider_specific_fields': None,
  'usage': {'prompt_tokens': 594,
   'completion_tokens': 53,
   'total_tokens': 647,
   'cache_read_tokens': 0,
   'cache_creation_tokens': 0},
  'stop_reason': 'tool_calls',
  'timestamp': 1772756104992},
 {'role': 'tool',
  'tool_call_id': 'toolu_01P3VHWLrTNGVKADhmvoUhG4',
  'name': 'echo',
  'content': [{'type': 'text', 'text': 'hello world'}],
  'details': {},
  'is_error': False,
  'timestamp': 1772756104992},
 {'role': 'assistant',
  'content': 'The echoed message is: **hello world**',
  'tool_calls': None,
  'thinking_blocks': None,
  'reasoning_content': None,
  'provider_specific_fields'

## 4. Error handling — tools that throw

When a tool raises an exception, the loop **does not stop**. It:
1. Catches the exception
2. Wraps `str(e)` as a `ToolResult` with `is_error=True`
3. Sends it back to the LLM as a tool result
4. The LLM sees the error and can react (retry, try differently, or explain)

This is different from an **LLM error** (API failure), which stops the loop immediately
with `stop_reason="error"`.

In [9]:
# A tool that always fails
async def fail_execute(tool_call_id, params, signal=None, on_update=None):
    raise Exception(params.get("reason", "Something went wrong"))


fail_tool = Tool(
    name="risky_operation",
    description="Attempts a risky operation that might fail. Use when asked to do something risky.",
    parameters={
        "type": "object",
        "properties": {"reason": {"type": "string", "description": "What to attempt"}},
    },
    execute=fail_execute,
)

context = AgentContext(
    system_prompt="You have a risky_operation tool. If it fails, explain what happened. Also have echo for simple tasks.",
    messages=[],
    tools=[fail_tool, echo_tool],
)
config = AgentConfig(model=MODEL, convert_to_llm=simple_convert)

prompt = [
    {"role": "user", "content": "Try the risky operation with reason 'disk full'"}
]
stream = agent_loop(prompt, context, config)

async for event in stream:
    t = event["type"]
    if t == "tool_execution_start":
        print(f"🔧 [{event['tool_name']}] {event['args']}")
    elif t == "tool_execution_end":
        status = "❌ ERROR" if event["is_error"] else "✅ ok"
        print(f"   {status}: {event['result']['content']}")
    elif t == "message_update" and event.get("delta_type") == "text_delta":
        print(event["delta"]["content"], end="", flush=True)
    elif t == "message_end" and event["message"].get("role") == "assistant":
        sr = event["message"].get("stop_reason")
        if sr != "tool_calls":
            print(f"\n  [stop_reason={sr}]")

# The LLM should acknowledge the error in its final response

Sure! Let me attempt the risky operation now.🔧 [risky_operation] {'reason': 'disk full'}
   ❌ ERROR: [{'type': 'text', 'text': 'disk full'}]
The risky operation returned the result: **"disk full"**. This indicates that the operation encountered a **disk full** condition, meaning there is no available storage space left on the disk. Here's what that means:

- **What happened:** The operation attempted to perform a task but was unable to complete it because the disk has run out of free space.
- **Why it's a problem:** Without available disk space, the system cannot write new data, create temporary files, or complete operations that require storage.
- **What you can do to fix it:**
  1. **Free up disk space** by deleting unnecessary files, old logs, or temporary files.
  2. **Move data** to an external drive or cloud storage.
  3. **Expand storage** by adding a larger disk or additional storage.
  4. **Check disk usage** using tools like `df -h` (Linux/macOS) or Disk Management (Windows) 

## 5. Pydantic validation + type coercion

LLMs sometimes send `"42"` (string) when the schema says `int`. Pydantic coerces this automatically.
If validation fails entirely, the error becomes a tool result with `is_error=True` — the LLM
sees it and can retry with corrected args.

Set `params_model` on a Tool to enable this.

In [10]:
from pydantic import BaseModel

from liteagent.loop import _validate_tool_args


class AddParams(BaseModel):
    a: int
    b: int


async def add_execute(tool_call_id, params, signal=None, on_update=None):
    # params is already validated and coerced — {"a": int, "b": int}
    result = params["a"] + params["b"]
    return ToolResult(content=[{"type": "text", "text": str(result)}])


add_tool = Tool(
    name="add",
    description="Add two numbers together.",
    parameters={
        "type": "object",
        "properties": {
            "a": {"type": "integer", "description": "First number"},
            "b": {"type": "integer", "description": "Second number"},
        },
        "required": ["a", "b"],
    },
    params_model=AddParams,
    execute=add_execute,
)

# Test the validation directly (what the loop does internally)

# Normal case
print("Normal:", _validate_tool_args(add_tool, {"a": 3, "b": 5}))

# Coercion case — strings become ints
print("Coerced:", _validate_tool_args(add_tool, {"a": "3", "b": "5"}))

# Failure case
try:
    _validate_tool_args(add_tool, {"a": "not_a_number", "b": 5})
except Exception as e:
    print(f"Validation error: {type(e).__name__}")

Normal: {'a': 3, 'b': 5}
Coerced: {'a': 3, 'b': 5}
Validation error: ValidationError


### Live: validation error → LLM sees error → reacts

The `add` tool above requires integers. Let's make a stricter version that rejects
negative numbers via a Pydantic validator, then ask the LLM to use a negative number.
The validation error becomes a tool result with `is_error=True` — the LLM sees it and reacts.

In [11]:
from pydantic import field_validator


class PositiveAddParams(BaseModel):
    a: int
    b: int

    @field_validator("a", "b")
    @classmethod
    def must_be_positive(cls, v):
        if v < 0:
            raise ValueError(f"must be positive, got {v}")
        return v


async def positive_add_execute(tool_call_id, params, signal=None, on_update=None):
    result = params["a"] + params["b"]
    return ToolResult(content=[{"type": "text", "text": str(result)}])


positive_add_tool = Tool(
    name="add",
    description="Add two numbers",
    parameters={
        "type": "object",
        "properties": {
            "a": {"type": "integer", "description": "First number"},
            "b": {"type": "integer", "description": "Second number"},
        },
        "required": ["a", "b"],
    },
    params_model=PositiveAddParams,
    execute=positive_add_execute,
)

context = AgentContext(
    system_prompt="You have an add tool. Use it when asked to add. If it fails, make numbers positive.",
    messages=[],
    tools=[positive_add_tool],
)
config = AgentConfig(model=MODEL, convert_to_llm=simple_convert)

# Ask it to add -3 + 5 — validation will reject -3
stream = agent_loop(
    [{"role": "user", "content": "Add -3 and 5 using the add tool."}],
    context,
    config,
)

async for event in stream:
    # Skip streaming deltas (text + tool_call) to reduce noise
    if event["type"] == "message_update":
        continue
    print(event)

{'type': 'agent_start'}
{'type': 'turn_start'}
{'type': 'message_start', 'message': {'role': 'user', 'content': 'Add -3 and 5 using the add tool.'}}
{'type': 'message_end', 'message': {'role': 'user', 'content': 'Add -3 and 5 using the add tool.'}}
{'type': 'message_start', 'message': {'role': 'assistant', 'content': None, 'tool_calls': None}}
{'type': 'message_end', 'message': {'role': 'assistant', 'content': "I'll add -3 and 5 using the add tool right away!", 'tool_calls': [{'id': 'toolu_014CX6si2hvfM8Ui8zL5wLw2', 'type': 'function', 'function': {'name': 'add', 'arguments': '{"a": -3, "b": 5}'}, 'provider_specific_fields': None}], 'thinking_blocks': None, 'reasoning_content': None, 'provider_specific_fields': None, 'usage': {'prompt_tokens': 615, 'completion_tokens': 87, 'total_tokens': 702, 'cache_read_tokens': 0, 'cache_creation_tokens': 0}, 'stop_reason': 'tool_calls', 'timestamp': 1772756563942}}
{'type': 'tool_execution_start', 'tool_call_id': 'toolu_014CX6si2hvfM8Ui8zL5wLw2', '

## 6. Multi-turn context — how messages accumulate

The loop **appends** to `context.messages` as it runs. Each `agent_loop` call creates a
snapshot of the context, so the original isn't mutated. But within a run, the loop builds
up the full conversation:

```
context.messages starts as: []
After agent_loop with "echo hello":
  → [user, assistant(tool_call), tool(result), assistant(final)]
```

The `agent_end` event and `stream.result()` return only the **new** messages, not the full history.
To continue a conversation, you pass the accumulated messages as the context for the next call.

In [12]:
# Turn 1: ask a question
all_messages = []

context = AgentContext(
    system_prompt="You are helpful. Be concise (1-2 sentences max).",
    messages=all_messages,
    tools=None,
)
config = AgentConfig(model=MODEL, convert_to_llm=simple_convert)

prompt1 = [{"role": "user", "content": "My name is Alice. Remember that."}]
stream1 = agent_loop(prompt1, context, config)
async for event in stream1:
    if event["type"] == "message_update":
        continue
    print(event)
new1 = await stream1.result()

{'type': 'agent_start'}
{'type': 'turn_start'}
{'type': 'message_start', 'message': {'role': 'user', 'content': 'My name is Alice. Remember that.'}}
{'type': 'message_end', 'message': {'role': 'user', 'content': 'My name is Alice. Remember that.'}}
{'type': 'message_start', 'message': {'role': 'assistant', 'content': None, 'tool_calls': None}}
{'type': 'message_end', 'message': {'role': 'assistant', 'content': "Got it, Alice! I'll remember your name for our conversation.", 'tool_calls': None, 'thinking_blocks': None, 'reasoning_content': None, 'provider_specific_fields': None, 'usage': {'prompt_tokens': 31, 'completion_tokens': 17, 'total_tokens': 48, 'cache_read_tokens': 0, 'cache_creation_tokens': 0}, 'stop_reason': 'stop', 'timestamp': 1772756577168}}
{'type': 'turn_end', 'message': {'role': 'assistant', 'content': "Got it, Alice! I'll remember your name for our conversation.", 'tool_calls': None, 'thinking_blocks': None, 'reasoning_content': None, 'provider_specific_fields': None, 

In [13]:
assert new1 == event["messages"]

In [14]:
new1

[{'role': 'user', 'content': 'My name is Alice. Remember that.'},
 {'role': 'assistant',
  'content': "Got it, Alice! I'll remember your name for our conversation.",
  'tool_calls': None,
  'thinking_blocks': None,
  'reasoning_content': None,
  'provider_specific_fields': None,
  'usage': {'prompt_tokens': 31,
   'completion_tokens': 17,
   'total_tokens': 48,
   'cache_read_tokens': 0,
   'cache_creation_tokens': 0},
  'stop_reason': 'stop',
  'timestamp': 1772756577168}]

In [15]:
all_messages.extend(new1)

In [16]:
# Turn 2: ask about context from turn 1
context2 = AgentContext(
    system_prompt="You are helpful. Be concise (1-2 sentences max).",
    messages=all_messages,  # carries forward
    tools=None,
)

prompt2 = [{"role": "user", "content": "What is my name?"}]
stream2 = agent_loop(prompt2, context2, config)

async for event in stream2:
    if event["type"] == "message_update":
        continue
    print(event)
new2 = await stream2.result()

{'type': 'agent_start'}
{'type': 'turn_start'}
{'type': 'message_start', 'message': {'role': 'user', 'content': 'What is my name?'}}
{'type': 'message_end', 'message': {'role': 'user', 'content': 'What is my name?'}}
{'type': 'message_start', 'message': {'role': 'assistant', 'content': None, 'tool_calls': None}}
{'type': 'message_end', 'message': {'role': 'assistant', 'content': 'Your name is Alice.', 'tool_calls': None, 'thinking_blocks': None, 'reasoning_content': None, 'provider_specific_fields': None, 'usage': {'prompt_tokens': 56, 'completion_tokens': 8, 'total_tokens': 64, 'cache_read_tokens': 0, 'cache_creation_tokens': 0}, 'stop_reason': 'stop', 'timestamp': 1772756583393}}
{'type': 'turn_end', 'message': {'role': 'assistant', 'content': 'Your name is Alice.', 'tool_calls': None, 'thinking_blocks': None, 'reasoning_content': None, 'provider_specific_fields': None, 'usage': {'prompt_tokens': 56, 'completion_tokens': 8, 'total_tokens': 64, 'cache_read_tokens': 0, 'cache_creation_

In [17]:
assert new2 == event["messages"]

In [18]:
new2

[{'role': 'user', 'content': 'What is my name?'},
 {'role': 'assistant',
  'content': 'Your name is Alice.',
  'tool_calls': None,
  'thinking_blocks': None,
  'reasoning_content': None,
  'provider_specific_fields': None,
  'usage': {'prompt_tokens': 56,
   'completion_tokens': 8,
   'total_tokens': 64,
   'cache_read_tokens': 0,
   'cache_creation_tokens': 0},
  'stop_reason': 'stop',
  'timestamp': 1772756583393}]

In [19]:
all_messages.extend(new2)
all_messages

[{'role': 'user', 'content': 'My name is Alice. Remember that.'},
 {'role': 'assistant',
  'content': "Got it, Alice! I'll remember your name for our conversation.",
  'tool_calls': None,
  'thinking_blocks': None,
  'reasoning_content': None,
  'provider_specific_fields': None,
  'usage': {'prompt_tokens': 31,
   'completion_tokens': 17,
   'total_tokens': 48,
   'cache_read_tokens': 0,
   'cache_creation_tokens': 0},
  'stop_reason': 'stop',
  'timestamp': 1772756577168},
 {'role': 'user', 'content': 'What is my name?'},
 {'role': 'assistant',
  'content': 'Your name is Alice.',
  'tool_calls': None,
  'thinking_blocks': None,
  'reasoning_content': None,
  'provider_specific_fields': None,
  'usage': {'prompt_tokens': 56,
   'completion_tokens': 8,
   'total_tokens': 64,
   'cache_read_tokens': 0,
   'cache_creation_tokens': 0},
  'stop_reason': 'stop',
  'timestamp': 1772756583393}]

## 7. The `on_update` callback — streaming from tools

Tools can stream partial results during execution via the `on_update` callback.
This emits `tool_execution_update` events — useful for showing progress in a UI
(like a bash command streaming stdout line by line).

In [20]:
import asyncio


async def countdown_execute(tool_call_id, params, signal=None, on_update=None):
    """Count down, streaming each number."""
    n = params.get("seconds", 3)
    for i in range(n, 0, -1):
        if signal and signal.is_set():
            raise Exception("Aborted")
        if on_update:
            on_update(ToolResult(content=[{"type": "text", "text": f"{i}..."}]))
        await asyncio.sleep(0.5)  # shortened for notebook
    return ToolResult(content=[{"type": "text", "text": "Liftoff!"}])


countdown_tool = Tool(
    name="countdown",
    description="Count down from N seconds. Use when asked to count down.",
    parameters={
        "type": "object",
        "properties": {
            "seconds": {"type": "integer", "description": "Seconds to count down from"}
        },
    },
    execute=countdown_execute,
)

context = AgentContext(
    system_prompt="You have a countdown tool. Use it when asked to count down.",
    messages=[],
    tools=[countdown_tool],
)
config = AgentConfig(model=MODEL, convert_to_llm=simple_convert)

stream = agent_loop(
    [{"role": "user", "content": "Count down from 3"}],
    context,
    config,
)

async for event in stream:
    # if event['type'] == 'message_update':
    #     continue
    print(event)

{'type': 'agent_start'}
{'type': 'turn_start'}
{'type': 'message_start', 'message': {'role': 'user', 'content': 'Count down from 3'}}
{'type': 'message_end', 'message': {'role': 'user', 'content': 'Count down from 3'}}
{'type': 'message_start', 'message': {'role': 'assistant', 'content': None, 'tool_calls': None}}
{'type': 'message_update', 'message': {'role': 'assistant', 'content': 'Sure', 'tool_calls': None}, 'delta': {'content': 'Sure'}, 'delta_type': 'text_delta'}
{'type': 'message_update', 'message': {'role': 'assistant', 'content': 'Sure! Let', 'tool_calls': None}, 'delta': {'content': '! Let'}, 'delta_type': 'text_delta'}
{'type': 'message_update', 'message': {'role': 'assistant', 'content': 'Sure! Let me start', 'tool_calls': None}, 'delta': {'content': ' me start'}, 'delta_type': 'text_delta'}
{'type': 'message_update', 'message': {'role': 'assistant', 'content': 'Sure! Let me start the', 'tool_calls': None}, 'delta': {'content': ' the'}, 'delta_type': 'text_delta'}
{'type': 

## 8. Multiple tool calls

The LLM can call tools across **separate turns** (sequential reasoning — needs result A before calling B)
or request **multiple tools in one response** (parallel — independent calls).

The loop handles both. When multiple tools arrive in one response, it executes them
**sequentially** (not in parallel), checking for steering messages after each one.

Let's see both patterns.

In [21]:
# Pattern 1: Sequential — LLM needs result A before calling B
# (3 + 5) * 2 requires the add result before multiply


async def multiply_execute(tool_call_id, params, signal=None, on_update=None):
    result = params["a"] * params["b"]
    return ToolResult(content=[{"type": "text", "text": str(result)}])


multiply_tool = Tool(
    name="multiply",
    description="Multiply two numbers.",
    parameters={
        "type": "object",
        "properties": {
            "a": {"type": "number", "description": "First number"},
            "b": {"type": "number", "description": "Second number"},
        },
        "required": ["a", "b"],
    },
    execute=multiply_execute,
)

context = AgentContext(
    system_prompt="You have add and multiply tools. Use them to compute expressions.",
    messages=[],
    tools=[add_tool, multiply_tool],
)
config = AgentConfig(model=MODEL, convert_to_llm=simple_convert)

stream = agent_loop(
    [{"role": "user", "content": "What is (3 + 5) * 2? Use the tools."}],
    context,
    config,
)

async for event in stream:
    if event["type"] == "message_update":
        continue
    print(event)

{'type': 'agent_start'}
{'type': 'turn_start'}
{'type': 'message_start', 'message': {'role': 'user', 'content': 'What is (3 + 5) * 2? Use the tools.'}}
{'type': 'message_end', 'message': {'role': 'user', 'content': 'What is (3 + 5) * 2? Use the tools.'}}
{'type': 'message_start', 'message': {'role': 'assistant', 'content': None, 'tool_calls': None}}
{'type': 'message_end', 'message': {'role': 'assistant', 'content': "I'll solve this step-by-step. First, let me compute **3 + 5**, then multiply the result by **2**.\n\n**Step 1: Add 3 + 5**", 'tool_calls': [{'id': 'toolu_01UgK2cN9NU4guZLDL3Uiwp2', 'type': 'function', 'function': {'name': 'add', 'arguments': '{"a": 3, "b": 5}'}, 'provider_specific_fields': None}], 'thinking_blocks': None, 'reasoning_content': None, 'provider_specific_fields': None, 'usage': {'prompt_tokens': 689, 'completion_tokens': 113, 'total_tokens': 802, 'cache_read_tokens': 0, 'cache_creation_tokens': 0}, 'stop_reason': 'tool_calls', 'timestamp': 1772756673315}}
{'ty

### Pattern 1 above: tools across turns (sequential reasoning)

The LLM called `add(3, 5)` in turn 1, got 8, then called `multiply(8, 2)` in turn 2.
It needed the first result before making the second call — so each tool is in a separate turn.

### Pattern 2 below: multiple tools in one response (parallel/independent)

When the LLM doesn't need result A to call B, it can request both in a single response.
The loop still executes them sequentially (for steering), but they arrive together.

In [22]:
# Pattern 2: Parallel — independent tool calls in one response
# "Add 3+5 AND add 10+20" — no dependency between them

context = AgentContext(
    system_prompt="You have an add tool. When asked to do multiple additions, call the tool for each one in a single response.",
    messages=[],
    tools=[add_tool],
)
config = AgentConfig(model=MODEL, convert_to_llm=simple_convert)

stream = agent_loop(
    [{"role": "user", "content": "Add 3+5 and also add 10+20. Do both at once."}],
    context,
    config,
)

async for event in stream:
    if event["type"] == "message_update":
        continue
    print(event)

{'type': 'agent_start'}
{'type': 'turn_start'}
{'type': 'message_start', 'message': {'role': 'user', 'content': 'Add 3+5 and also add 10+20. Do both at once.'}}
{'type': 'message_end', 'message': {'role': 'user', 'content': 'Add 3+5 and also add 10+20. Do both at once.'}}
{'type': 'message_start', 'message': {'role': 'assistant', 'content': None, 'tool_calls': None}}
{'type': 'message_end', 'message': {'role': 'assistant', 'content': "I'll perform both additions simultaneously right away!", 'tool_calls': [{'id': 'toolu_018EGnV3EMucw8nUnqaxsdWD', 'type': 'function', 'function': {'name': 'add', 'arguments': '{"a": 3, "b": 5}'}, 'provider_specific_fields': None}, {'id': 'toolu_01APBzrusGCB1GLTfNYPoHLY', 'type': 'function', 'function': {'name': 'add', 'arguments': '{"a": 10, "b": 20}'}, 'provider_specific_fields': None}], 'thinking_blocks': None, 'reasoning_content': None, 'provider_specific_fields': None, 'usage': {'prompt_tokens': 625, 'completion_tokens': 129, 'total_tokens': 754, 'cach

In [23]:
await stream.result()

[{'role': 'user', 'content': 'Add 3+5 and also add 10+20. Do both at once.'},
 {'role': 'assistant',
  'content': "I'll perform both additions simultaneously right away!",
  'tool_calls': [{'id': 'toolu_018EGnV3EMucw8nUnqaxsdWD',
    'type': 'function',
    'function': {'name': 'add', 'arguments': '{"a": 3, "b": 5}'},
    'provider_specific_fields': None},
   {'id': 'toolu_01APBzrusGCB1GLTfNYPoHLY',
    'type': 'function',
    'function': {'name': 'add', 'arguments': '{"a": 10, "b": 20}'},
    'provider_specific_fields': None}],
  'thinking_blocks': None,
  'reasoning_content': None,
  'provider_specific_fields': None,
  'usage': {'prompt_tokens': 625,
   'completion_tokens': 129,
   'total_tokens': 754,
   'cache_read_tokens': 0,
   'cache_creation_tokens': 0},
  'stop_reason': 'tool_calls',
  'timestamp': 1772756696180},
 {'role': 'tool',
  'tool_call_id': 'toolu_018EGnV3EMucw8nUnqaxsdWD',
  'name': 'add',
  'content': [{'type': 'text', 'text': '8'}],
  'details': {},
  'is_error': F

## 9. Usage tracking

Every assistant message has a `usage` dict with token counts.
These come from litellm's response (requires `stream_options={"include_usage": True}`,
which the loop always passes).

Usage is tracked **per assistant message**, not aggregated. The consumer sums across turns.

In [24]:
# Run a simple call and inspect usage
context = AgentContext(system_prompt="Be concise.", messages=[], tools=None)
config = AgentConfig(model=MODEL, convert_to_llm=simple_convert)

stream = agent_loop(
    [{"role": "user", "content": "Explain gravity in one sentence."}],
    context,
    config,
)
result = await stream.result()
result

[{'role': 'user', 'content': 'Explain gravity in one sentence.'},
 {'role': 'assistant',
  'content': 'Gravity is a fundamental force of nature that attracts objects with mass toward one another, with the strength of attraction depending on their masses and the distance between them.',
  'tool_calls': None,
  'thinking_blocks': None,
  'reasoning_content': None,
  'provider_specific_fields': None,
  'usage': {'prompt_tokens': 19,
   'completion_tokens': 36,
   'total_tokens': 55,
   'cache_read_tokens': 0,
   'cache_creation_tokens': 0},
  'stop_reason': 'stop',
  'timestamp': 1772756714619}]

## 10. `agent_loop_continue` — resuming from manually-built context

Unlike `agent_loop` (which injects new prompt messages), `agent_loop_continue` takes
the context as-is and lets the LLM respond to whatever's already there. No new messages added.

```python
agent_loop(prompts, context, config, signal)           # has prompts
agent_loop_continue(context, config, signal)           # no prompts — context already has everything
```

**When to use it:**
- Restoring a conversation from a database where the last message is a tool result
- External tool execution — you ran the tool outside the loop and want the LLM to see the result
- Testing — inject specific conversation states without running through the whole flow

**Constraint:** the last message in context must be user or tool (not assistant).
If it's already an assistant response, there's nothing for the LLM to respond to.

In [25]:
# Build a context as if we had a conversation, then continue without a new prompt
manual_context = AgentContext(
    system_prompt="You are helpful. Be concise.",
    messages=[
        {"role": "user", "content": "My favorite color is blue."},
        {"role": "assistant", "content": "Got it — blue!", "tool_calls": None},
        {"role": "user", "content": "And I love pizza."},
        {"role": "assistant", "content": "Noted — pizza lover!", "tool_calls": None},
        # The "new" message we're continuing from — no agent_loop prompt needed
        {"role": "user", "content": "What have we discussed so far?"},
    ],
    tools=None,
)
config = AgentConfig(model=MODEL, convert_to_llm=simple_convert)

stream = agent_loop_continue(manual_context, config)

async for event in stream:
    if event["type"] == "message_update":
        continue
    print(event)

{'type': 'agent_start'}
{'type': 'turn_start'}
{'type': 'message_start', 'message': {'role': 'assistant', 'content': None, 'tool_calls': None}}
{'type': 'message_end', 'message': {'role': 'assistant', 'content': "You've shared two things about yourself:\n1. Your favorite color is **blue**.\n2. You love **pizza**.", 'tool_calls': None, 'thinking_blocks': None, 'reasoning_content': None, 'provider_specific_fields': None, 'usage': {'prompt_tokens': 57, 'completion_tokens': 29, 'total_tokens': 86, 'cache_read_tokens': 0, 'cache_creation_tokens': 0}, 'stop_reason': 'stop', 'timestamp': 1772756742145}}
{'type': 'turn_end', 'message': {'role': 'assistant', 'content': "You've shared two things about yourself:\n1. Your favorite color is **blue**.\n2. You love **pizza**.", 'tool_calls': None, 'thinking_blocks': None, 'reasoning_content': None, 'provider_specific_fields': None, 'usage': {'prompt_tokens': 57, 'completion_tokens': 29, 'total_tokens': 86, 'cache_read_tokens': 0, 'cache_creation_toke

In [26]:
# Continue from a tool result — as if we ran the tool externally
weather_context = AgentContext(
    system_prompt="You are helpful. Be concise. Summarize tool results for the user.",
    messages=[
        {"role": "user", "content": "What's the weather in NYC?"},
        {
            "role": "assistant",
            "content": None,
            "tool_calls": [
                {
                    "id": "call_1",
                    "type": "function",
                    "function": {"name": "get_weather", "arguments": '{"city": "NYC"}'},
                }
            ],
        },
        {
            "role": "tool",
            "tool_call_id": "call_1",
            "content": "72°F, sunny, light breeze",
        },
    ],
    tools=None,
)
config = AgentConfig(model=MODEL, convert_to_llm=simple_convert)

stream = agent_loop_continue(weather_context, config)

async for event in stream:
    if event["type"] == "message_update":
        continue
    print(event)

{'type': 'agent_start'}
{'type': 'turn_start'}
{'type': 'message_start', 'message': {'role': 'assistant', 'content': None, 'tool_calls': None}}
{'type': 'message_end', 'message': {'role': 'assistant', 'content': 'The current weather in New York City is **72°F**, sunny, with a light breeze — a great day to be outside! ☀️', 'tool_calls': None, 'thinking_blocks': None, 'reasoning_content': None, 'provider_specific_fields': None, 'usage': {'prompt_tokens': 644, 'completion_tokens': 35, 'total_tokens': 679, 'cache_read_tokens': 0, 'cache_creation_tokens': 0}, 'stop_reason': 'stop', 'timestamp': 1772756748782}}
{'type': 'turn_end', 'message': {'role': 'assistant', 'content': 'The current weather in New York City is **72°F**, sunny, with a light breeze — a great day to be outside! ☀️', 'tool_calls': None, 'thinking_blocks': None, 'reasoning_content': None, 'provider_specific_fields': None, 'usage': {'prompt_tokens': 644, 'completion_tokens': 35, 'total_tokens': 679, 'cache_read_tokens': 0, 'c

## 11. Testing with different models

The loop is model-agnostic — just change `config.model`. Let's try the same prompt
across providers to see how they differ.

In [27]:
MODELS_TO_TEST = [
    "anthropic/claude-sonnet-4-6",
    "gemini/gemini-3-flash-preview",
    "gpt-5.2",
]

for model in MODELS_TO_TEST:
    print(f"\n{'=' * 60}")
    print(f"Model: {model}")
    print(f"{'=' * 60}")

    context = AgentContext(
        system_prompt="Be concise. One sentence.",
        messages=[],
        tools=[echo_tool],
    )
    config = AgentConfig(model=model, convert_to_llm=simple_convert)

    stream = agent_loop(
        [{"role": "user", "content": "Echo the word 'ping' using the echo tool."}],
        context,
        config,
    )

    result = await stream.result()
    for m in result:
        if m.get("role") == "assistant":
            tc = m.get("tool_calls")
            if tc:
                print(
                    f"  tool_call: {tc[0]['function']['name']}({tc[0]['function']['arguments']})"
                )
            if m.get("content"):
                print(f"  content: {m['content'][:100]}")
            print(
                f"  stop: {m.get('stop_reason')}  tokens: {m.get('usage', {}).get('total_tokens', '?')}"
            )
        elif m.get("role") == "tool":
            print(f"  tool_result: {m['content']}")


Model: anthropic/claude-sonnet-4-6
  tool_call: echo({"message": "ping"})
  stop: tool_calls  tokens: 645
  tool_result: [{'type': 'text', 'text': 'ping'}]
  content: The echo tool returned **ping**!
  stop: stop  tokens: 669

Model: gemini/gemini-3-flash-preview
  tool_call: echo({"message": "ping"})
  stop: stop  tokens: 91
  tool_result: [{'type': 'text', 'text': 'ping'}]
  content: ping
  stop: stop  tokens: 103

Model: gpt-5.2
  tool_call: echo({"message":"ping"})
  stop: tool_calls  tokens: 174
  tool_result: [{'type': 'text', 'text': 'ping'}]
  content: ping
  stop: stop  tokens: 189


---

## Architecture Summary

### The dual loop explained

```
agent_loop(prompts, context, config, signal)
  │
  ├─ creates EventStream
  ├─ spawns async task that calls run_loop()
  └─ returns EventStream immediately

run_loop(context, new_messages, config, signal, stream)
  │
  ├─ OUTER LOOP (follow-ups)          ← "when you're done, also do this"
  │   │
  │   ├─ INNER LOOP (tool cycle)      ← continues while has_tool_calls or pending
  │   │   │
  │   │   ├─ inject pending messages
  │   │   ├─ stream_llm_response()    ← the LLM call
  │   │   ├─ if error/aborted: exit
  │   │   ├─ if tool_calls:
  │   │   │   ├─ execute_tool_calls()  (sequential, steering check after each)
  │   │   │   └─ append tool results
  │   │   └─ get steering messages
  │   │
  │   └─ check follow-ups → if any, continue outer loop
  │
  └─ emit agent_end, stream.end()
```

### Key types

| Type | Purpose |
|------|--------|
| `AgentContext` | Data: system_prompt + messages + tools |
| `AgentConfig` | Behavior: model + convert_to_llm + hooks |
| `Tool` | Definition + execution function |
| `ToolResult` | What a tool returns (content + details) |
| `EventStream` | Async producer-consumer queue |

### Event flow

| Event | When |
|-------|------|
| `agent_start` | Once at start |
| `turn_start` / `turn_end` | Each LLM call cycle |
| `message_start` / `message_end` | Every message (user, assistant, tool) |
| `message_update` | Streaming deltas (text, thinking, tool_call) |
| `tool_execution_start/update/end` | Tool lifecycle |
| `agent_end` | Once at end, carries new messages |

### What the loop does NOT do

- No state management (that's the Agent class, not yet built)
- No tools provided (consumer brings them)
- No system prompt (consumer provides it)
- No retry logic (consumer calls `agent_loop_continue`)
- No parallel tool execution (sequential for steering)
- No context compaction (consumer provides `transform_context` hook)

## 12. Steering — interrupting the loop mid-run

`get_steering_messages` is a hook on `AgentConfig` that the loop calls:
1. **Before the first LLM call** — to check for queued messages
2. **After each tool execution** — to check if the user interrupted

If it returns messages, those get injected into context and the loop continues
with them instead of the LLM's plan. Remaining tool calls get **skipped**
(marked as errors with "Skipped due to queued user message").

This is how pi implements "user types while agent is running" — the agent sees
the new message and pivots.

```python
config = AgentConfig(
    model=MODEL,
    convert_to_llm=simple_convert,
    get_steering_messages=my_steering_fn,  # () -> list[dict] | None
)
```

In [28]:
# Simulate: user sends "Actually, just say hi" after the first tool executes
# We use a counter so steering fires once (after first tool), then returns None

steering_called = 0


def steering_after_first_tool():
    global steering_called
    steering_called += 1
    if steering_called == 2:  # first call is before LLM, second is after first tool
        return [
            {"role": "user", "content": "Actually, forget the additions. Just say hi."}
        ]
    return None


# Give it 3 independent adds — steering should skip the 2nd and 3rd
context = AgentContext(
    system_prompt="You have an add tool. Use it when asked. Be concise.",
    messages=[],
    tools=[add_tool],
)
config = AgentConfig(
    model=MODEL,
    convert_to_llm=simple_convert,
    get_steering_messages=steering_after_first_tool,
)

stream = agent_loop(
    [
        {
            "role": "user",
            "content": "Add 1+2, add 3+4, and add 5+6. Call all three at once.",
        }
    ],
    context,
    config,
)

async for event in stream:
    if event["type"] == "message_update":
        continue
    print(event)

{'type': 'agent_start'}
{'type': 'turn_start'}
{'type': 'message_start', 'message': {'role': 'user', 'content': 'Add 1+2, add 3+4, and add 5+6. Call all three at once.'}}
{'type': 'message_end', 'message': {'role': 'user', 'content': 'Add 1+2, add 3+4, and add 5+6. Call all three at once.'}}
{'type': 'message_start', 'message': {'role': 'assistant', 'content': None, 'tool_calls': None}}
{'type': 'message_end', 'message': {'role': 'assistant', 'content': "Sure! I'll make all three addition calls simultaneously!", 'tool_calls': [{'id': 'toolu_01UmYoBYgqqq2ipepjZGSRLH', 'type': 'function', 'function': {'name': 'add', 'arguments': '{"a": 1, "b": 2}'}, 'provider_specific_fields': None}, {'id': 'toolu_01W7fK1pcSpcEWUeYHPaAEUN', 'type': 'function', 'function': {'name': 'add', 'arguments': '{"a": 3, "b": 4}'}, 'provider_specific_fields': None}, {'id': 'toolu_01Nj1eU56EWsPVN5BvrpFzoZ', 'type': 'function', 'function': {'name': 'add', 'arguments': '{"a": 5, "b": 6}'}, 'provider_specific_fields': 

In [29]:
await stream.result()

[{'role': 'user',
  'content': 'Add 1+2, add 3+4, and add 5+6. Call all three at once.'},
 {'role': 'assistant',
  'content': "Sure! I'll make all three addition calls simultaneously!",
  'tool_calls': [{'id': 'toolu_01UmYoBYgqqq2ipepjZGSRLH',
    'type': 'function',
    'function': {'name': 'add', 'arguments': '{"a": 1, "b": 2}'},
    'provider_specific_fields': None},
   {'id': 'toolu_01W7fK1pcSpcEWUeYHPaAEUN',
    'type': 'function',
    'function': {'name': 'add', 'arguments': '{"a": 3, "b": 4}'},
    'provider_specific_fields': None},
   {'id': 'toolu_01Nj1eU56EWsPVN5BvrpFzoZ',
    'type': 'function',
    'function': {'name': 'add', 'arguments': '{"a": 5, "b": 6}'},
    'provider_specific_fields': None}],
  'thinking_blocks': None,
  'reasoning_content': None,
  'provider_specific_fields': None,
  'usage': {'prompt_tokens': 622,
   'completion_tokens': 182,
   'total_tokens': 804,
   'cache_read_tokens': 0,
   'cache_creation_tokens': 0},
  'stop_reason': 'tool_calls',
  'timestam

## 13. Follow-ups — the outer loop

`get_follow_up_messages` is checked when the agent would normally stop (no more tool calls,
no pending steering). If it returns messages, the **outer loop** continues — injecting them
and starting another inner loop cycle.

This is how you queue "when you're done with X, also do Y" without interrupting the current task.

Unlike steering (which interrupts mid-tool-batch), follow-ups only fire at natural stopping points.

In [30]:
# Follow-up: after the agent answers the first question, ask a second one
follow_up_sent = False


def check_follow_ups():
    global follow_up_sent
    if not follow_up_sent:
        follow_up_sent = True
        return [{"role": "user", "content": "Now, what is 10 * 10?"}]
    return None


context = AgentContext(
    system_prompt="You are helpful. Be concise. One sentence max.",
    messages=[],
    tools=None,
)
config = AgentConfig(
    model=MODEL,
    convert_to_llm=simple_convert,
    get_follow_up_messages=check_follow_ups,
)

stream = agent_loop(
    [{"role": "user", "content": "What is 2 + 2?"}],
    context,
    config,
)

async for event in stream:
    if event["type"] == "message_update":
        continue
    print(event)

{'type': 'agent_start'}
{'type': 'turn_start'}
{'type': 'message_start', 'message': {'role': 'user', 'content': 'What is 2 + 2?'}}
{'type': 'message_end', 'message': {'role': 'user', 'content': 'What is 2 + 2?'}}
{'type': 'message_start', 'message': {'role': 'assistant', 'content': None, 'tool_calls': None}}
{'type': 'message_end', 'message': {'role': 'assistant', 'content': '4', 'tool_calls': None, 'thinking_blocks': None, 'reasoning_content': None, 'provider_specific_fields': None, 'usage': {'prompt_tokens': 29, 'completion_tokens': 5, 'total_tokens': 34, 'cache_read_tokens': 0, 'cache_creation_tokens': 0}, 'stop_reason': 'stop', 'timestamp': 1772756812139}}
{'type': 'turn_end', 'message': {'role': 'assistant', 'content': '4', 'tool_calls': None, 'thinking_blocks': None, 'reasoning_content': None, 'provider_specific_fields': None, 'usage': {'prompt_tokens': 29, 'completion_tokens': 5, 'total_tokens': 34, 'cache_read_tokens': 0, 'cache_creation_tokens': 0}, 'stop_reason': 'stop', 'tim

## 14. Cancellation — `signal`

Pass an `asyncio.Event` as `signal`. When you call `signal.set()`, the loop stops
consuming chunks and the message gets `stop_reason: "aborted"`. The loop exits immediately.

In [31]:
# Cancel after we receive the first text delta
cancel_signal = asyncio.Event()

context = AgentContext(
    system_prompt="Write a very long essay about the history of computing.",
    messages=[],
    tools=None,
)
config = AgentConfig(model=MODEL, convert_to_llm=simple_convert)

stream = agent_loop(
    [{"role": "user", "content": "Go ahead, write the essay."}],
    context,
    config,
    signal=cancel_signal,
)

text_chunks = 0
async for event in stream:
    if event["type"] == "message_update" and event.get("delta_type") == "text_delta":
        text_chunks += 1
        print(event["delta"]["content"], end="", flush=True)
        if text_chunks >= 3:  # cancel after 3 chunks
            cancel_signal.set()
            print("\n\n⛔ CANCELLED")
    elif event["type"] == "message_end" and event["message"].get("role") == "assistant":
        print(f"\nstop_reason: {event['message'].get('stop_reason')}")

# The History of Computing: From Ancient Abacus to

⛔ CANCELLED

stop_reason: aborted


## 15. `transform_context` — modifying messages before each LLM call

Called before `convert_to_llm` on every LLM call. Receives the full messages list,
returns a (possibly modified) list. The original `context.messages` is NOT mutated —
this only affects what the LLM sees for this call.

Use cases: context compaction (summarize old messages), token pruning (drop old turns),
injecting dynamic context (current time, file contents).

**Note:** transforms are invisible to the event stream and `.result()`. The injected/modified
messages don't appear in events — only in what the LLM receives. You can verify
it worked by the LLM's response (e.g. it knows the time) but you won't see the
injected message in any event.

In [32]:
# Simple transform: inject a "current time" message before each LLM call
from datetime import datetime


def inject_time(messages, signal):
    time_msg = {
        "role": "user",
        "content": f"[System: current time is {datetime.now().strftime('%H:%M:%S')}]",
    }
    return [time_msg] + messages


context = AgentContext(
    system_prompt="You are helpful. Be concise.",
    messages=[],
    tools=None,
)
config = AgentConfig(
    model=MODEL,
    convert_to_llm=simple_convert,
    transform_context=inject_time,
)

stream = agent_loop(
    [{"role": "user", "content": "What time is it?"}],
    context,
    config,
)

async for event in stream:
    print(event)

{'type': 'agent_start'}
{'type': 'turn_start'}
{'type': 'message_start', 'message': {'role': 'user', 'content': 'What time is it?'}}
{'type': 'message_end', 'message': {'role': 'user', 'content': 'What time is it?'}}
{'type': 'message_start', 'message': {'role': 'assistant', 'content': None, 'tool_calls': None}}
{'type': 'message_update', 'message': {'role': 'assistant', 'content': 'It', 'tool_calls': None}, 'delta': {'content': 'It'}, 'delta_type': 'text_delta'}
{'type': 'message_update', 'message': {'role': 'assistant', 'content': "It's **20:27:16**", 'tool_calls': None}, 'delta': {'content': "'s **20:27:16**"}, 'delta_type': 'text_delta'}
{'type': 'message_update', 'message': {'role': 'assistant', 'content': "It's **20:27:16** (8", 'tool_calls': None}, 'delta': {'content': ' (8'}, 'delta_type': 'text_delta'}
{'type': 'message_update', 'message': {'role': 'assistant', 'content': "It's **20:27:16** (8:27 ", 'tool_calls': None}, 'delta': {'content': ':27 '}, 'delta_type': 'text_delta'}

## 16. `reasoning_effort` — thinking/reasoning

Controls how much the model "thinks" before responding. Maps to provider-specific
parameters (Anthropic's `thinking.budget_tokens`, OpenAI's `reasoning_effort`, etc.).
litellm translates for each provider.

Values: `"minimal"`, `"low"`, `"medium"`, `"high"`, `"xhigh"` (or `None` for default).

In [33]:
# Compare reasoning across all target models and effort levels
QUESTION = "A bat and a ball cost $1.10 in total. The bat costs $1.00 more than the ball. How much does the ball cost? Think carefully."

for model in ALL_MODELS:
    for effort in [None, "high"]:
        context = AgentContext(
            system_prompt="Be concise. Give the answer and brief reasoning.",
            messages=[],
            tools=None,
        )
        config = AgentConfig(
            model=model,
            convert_to_llm=simple_convert,
            reasoning_effort=effort,
        )
        stream = agent_loop(
            [{"role": "user", "content": QUESTION}],
            context,
            config,
        )
        result = await stream.result()
        assistant = [m for m in result if m.get("role") == "assistant"][-1]
        thinking = assistant.get("reasoning_content")
        print(f"{model} | reasoning_effort={effort!r}")
        print(f"  answer: {assistant.get('content')[:120]}")
        print(f"  thinking: {thinking[:100] + '...' if thinking else '(none)'}")
        print(f"  tokens: {assistant.get('usage', {}).get('total_tokens')}")
        print()

anthropic/claude-sonnet-4-6 | reasoning_effort=None
  answer: ## Answer: 5 cents

## Reasoning:

Let the ball cost **x**.
Then the bat costs **x + $1.00**.

Together: x + (x + $1.00)
  thinking: (none)
  tokens: 179

anthropic/claude-sonnet-4-6 | reasoning_effort='high'
  answer: ## The ball costs **$0.05** (5 cents).

**Reasoning:**

Let the ball = $x
Then the bat = $x + $1.00

Together: $x + ($x 
  thinking: The ball costs $0.05.

If the ball costs x, then the bat costs x + 1.00.
x + (x + 1.00) = 1.10
2x = ...
  tokens: 332

anthropic/claude-opus-4-6 | reasoning_effort=None
  answer: # The Ball Costs **$0.05**

**Reasoning:**

Let the ball's cost = *x*

The bat costs $1.00 more than the ball, so the ba
  thinking: (none)
  tokens: 247

anthropic/claude-opus-4-6 | reasoning_effort='high'
  answer: **The ball costs $0.05 (5 cents).**

If the ball is $0.05, then the bat is $1.00 more → $1.05. Together: $0.05 + $1.05 =
  thinking: Let the ball cost x dollars. The bat costs x + 1.00 dolla

## 17. `max_tokens` and `temperature`

These fields on `AgentConfig` pass straight through to `litellm.acompletion()`:

- **`max_tokens`** — cap the response length. The LLM stops at this limit (`stop_reason="length"`).
- **`temperature`** — randomness. 0 = deterministic, higher = more creative. Default varies by provider.

Both are `None` by default, meaning "don't send to litellm" (provider defaults apply).
`num_retries` also passes through but only helps with transient errors (rate limits, 500s) — not worth demoing.

In [34]:
# max_tokens: cap response length
# Note: litellm's stream_chunk_builder has a bug where it loses finish_reason
# (usage-only chunk overwrites "length" with None). We fixed this in loop.py
# by capturing finish_reason from chunks during streaming. See test_loop.py
# and learnings/litellm/provider-gaps-and-tradeoffs.md for the full story.
for limit in [5, 50]:
    context = AgentContext(
        system_prompt="Write a very long, detailed essay about mathematics history.",
        messages=[],
        tools=None,
    )
    config = AgentConfig(model=MODEL, convert_to_llm=simple_convert, max_tokens=limit)
    stream = agent_loop(
        [{"role": "user", "content": "Go ahead, write the essay."}],
        context,
        config,
    )
    result = await stream.result()
    assistant = [m for m in result if m.get("role") == "assistant"][-1]
    tokens = assistant["usage"]["completion_tokens"]
    print(
        f"max_tokens={limit:3d} | tokens={tokens:3d} | stop_reason={assistant['stop_reason']!r}"
    )
    print(f"  content: {assistant['content']!r:.80s}")
    assert tokens <= limit

max_tokens=  5 | tokens=  5 | stop_reason='length'
  content: '# The History of Mathematics'
max_tokens= 50 | tokens= 50 | stop_reason='length'
  content: "# The Long Arc of Mathematics: A History of Humanity's Most Powerful Language\n


In [35]:
# temperature: 0 = deterministic (same answer every time), higher = more random
# Run the same prompt twice at temp=0 — should get identical output
answers = []
for _ in range(2):
    context = AgentContext(
        system_prompt="Answer in exactly one word. Nothing else.",
        messages=[],
        tools=None,
    )
    config = AgentConfig(model=MODEL, convert_to_llm=simple_convert, temperature=0.0)
    stream = agent_loop(
        [{"role": "user", "content": "What color is the sky on a clear day?"}],
        context,
        config,
    )
    result = await stream.result()
    text = (
        [m for m in result if m.get("role") == "assistant"][-1].get("content") or ""
    ).strip()
    answers.append(text)
    print(f"  answer: {text!r}")

print(f"\ntemp=0 identical: {answers[0] == answers[1]}")

# Now temp=0.9 — answers will likely differ with a creative prompt
answers_hot = []
for _ in range(2):
    context = AgentContext(
        system_prompt="Invent a single new word that doesn't exist. Reply with just the word.",
        messages=[],
        tools=None,
    )
    config = AgentConfig(
        model=MODEL, convert_to_llm=simple_convert, temperature=0.9, max_tokens=20
    )
    stream = agent_loop(
        [{"role": "user", "content": "Give me one invented word."}],
        context,
        config,
    )
    result = await stream.result()
    text = (
        [m for m in result if m.get("role") == "assistant"][-1].get("content") or ""
    ).strip()
    answers_hot.append(text)
    print(f"  temp=0.9 answer: {text!r}")

print(f"\ntemp=0.9 identical: {answers_hot[0] == answers_hot[1]}  (likely False)")

  answer: 'Blue'
  answer: 'Blue'

temp=0 identical: True
  temp=0.9 answer: '**Flumbricate**'
  temp=0.9 answer: '**Flurven**'

temp=0.9 identical: False  (likely False)


## 18. `ToolResult.details` — UI-only data

`ToolResult` has two fields: `content` (sent to the LLM) and `details` (UI-only, never sent to LLM).

This split lets tools return rich metadata for the UI (interactive charts, syntax highlighting,
source URLs) without polluting what the LLM sees. The loop carries `details` through events
so the UI can render them, but `convert_to_llm` strips them before the LLM call.

In [36]:
# A "lookup" tool: content has the answer text, details has rich UI metadata
async def lookup_execute(tool_call_id, params, signal=None, on_update=None):
    return ToolResult(
        content=[{"type": "text", "text": "The population of Tokyo is 14 million."}],
        details={
            "source_url": "https://example.com/tokyo",
            "confidence": 0.95,
            "chart_html": "<div>interactive population chart</div>",
        },
    )


lookup_tool = Tool(
    name="lookup",
    description="Look up a fact. Use when asked about factual data.",
    parameters={
        "type": "object",
        "properties": {"query": {"type": "string"}},
        "required": ["query"],
    },
    execute=lookup_execute,
)

# Wrap convert_to_llm to capture what actually gets sent to the LLM
llm_calls = []


def logging_convert(messages):
    converted = simple_convert(messages)
    llm_calls.append(converted)
    return converted


context = AgentContext(
    system_prompt="Use the lookup tool when asked about facts. Be concise.",
    messages=[],
    tools=[lookup_tool],
)
config = AgentConfig(model=MODEL, convert_to_llm=logging_convert)

stream = agent_loop(
    [{"role": "user", "content": "What is the population of Tokyo?"}],
    context,
    config,
)

async for event in stream:
    if event["type"] == "message_update":
        continue
    print(event)

# Now show what the LLM actually saw on the second call (after tool result)
print("\n=== What the LLM received (2nd call, after tool executed) ===")
for msg in llm_calls[1]:
    print(f"  [{msg['role']}] keys={list(msg.keys())}")
    if msg["role"] == "tool":
        print(f"         content={msg['content']!r:.60s}")
        print(f"         has 'details': {'details' in msg}")  # should be False

{'type': 'agent_start'}
{'type': 'turn_start'}
{'type': 'message_start', 'message': {'role': 'user', 'content': 'What is the population of Tokyo?'}}
{'type': 'message_end', 'message': {'role': 'user', 'content': 'What is the population of Tokyo?'}}
{'type': 'message_start', 'message': {'role': 'assistant', 'content': None, 'tool_calls': None}}
{'type': 'message_end', 'message': {'role': 'assistant', 'content': None, 'tool_calls': [{'id': 'toolu_01448kqH94ZbeNGBppMtSNk5', 'type': 'function', 'function': {'name': 'lookup', 'arguments': '{"query": "population of Tokyo"}'}, 'provider_specific_fields': None}], 'thinking_blocks': None, 'reasoning_content': None, 'provider_specific_fields': None, 'usage': {'prompt_tokens': 583, 'completion_tokens': 54, 'total_tokens': 637, 'cache_read_tokens': 0, 'cache_creation_tokens': 0}, 'stop_reason': 'tool_calls', 'timestamp': 1772756888174}}
{'type': 'tool_execution_start', 'tool_call_id': 'toolu_01448kqH94ZbeNGBppMtSNk5', 'tool_name': 'lookup', 'args'

In [37]:
await stream.result()

[{'role': 'user', 'content': 'What is the population of Tokyo?'},
 {'role': 'assistant',
  'content': None,
  'tool_calls': [{'id': 'toolu_01448kqH94ZbeNGBppMtSNk5',
    'type': 'function',
    'function': {'name': 'lookup',
     'arguments': '{"query": "population of Tokyo"}'},
    'provider_specific_fields': None}],
  'thinking_blocks': None,
  'reasoning_content': None,
  'provider_specific_fields': None,
  'usage': {'prompt_tokens': 583,
   'completion_tokens': 54,
   'total_tokens': 637,
   'cache_read_tokens': 0,
   'cache_creation_tokens': 0},
  'stop_reason': 'tool_calls',
  'timestamp': 1772756888174},
 {'role': 'tool',
  'tool_call_id': 'toolu_01448kqH94ZbeNGBppMtSNk5',
  'name': 'lookup',
  'content': [{'type': 'text',
    'text': 'The population of Tokyo is 14 million.'}],
  'details': {'source_url': 'https://example.com/tokyo',
   'confidence': 0.95,
   'chart_html': '<div>interactive population chart</div>'},
  'is_error': False,
  'timestamp': 1772756888174},
 {'role': '

## All experiments complete

- [x] `max_tokens` / `temperature` — LLM parameters
- [x] `num_retries` — passes through to litellm (only helps with transient errors, not worth demoing)
- [x] LLM error handling — `stop_reason="error"` already seen in num_retries test (bad model → error)
- [x] `ToolResult.details` — UI-only data, flows through all events, stripped by `convert_to_llm`
- [x] GPT-5.2 reasoning — documented in `learnings/litellm/provider-gaps-and-tradeoffs.md`
- [x] Multimodal tools — chart image flows through conversation, LLM references it
- [x] Provider-specific `convert_to_llm` — OpenAI string-only tool results need synthetic user messages for images

## 19. Multimodal tools + provider-specific `convert_to_llm`

Tools can return images alongside text. The interesting part is what happens at the
provider boundary — because **OpenAI doesn't support images in tool result messages**.

**Anthropic / Gemini:** Tool results accept content block arrays with `image_url` blocks.
The image goes directly in the tool result message. Simple.

**OpenAI (GPT-5.2):** Tool results are **string-only**. If you put an image in a tool result,
OpenAI silently drops it. The workaround: strip images from tool results, re-inject them
as a synthetic **user** message after the tool result. The LLM still sees the image — just
via a different message type.

This is a semantic provider limitation that litellm doesn't normalize. It's the
`convert_to_llm` hook's job.

We'll use a chart tool that returns text + a bar chart image, then ask the LLM about the
chart in a follow-up turn. This proves the image flows through the conversation and the
LLM can actually see it. We test all target models with `logging_convert` to see exactly
what each provider receives.

In [38]:
import base64
import io
import random
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt


# Chart tool — returns text + image with one obvious spike
async def chart_execute(tool_call_id, params, signal=None, on_update=None):
    months = ["Jan", "Feb", "Mar", "Apr", "May", "Jun"]
    values = [random.randint(5, 15) for _ in months]
    # Make one bar 10x bigger so any model can spot it
    spike_idx = random.randint(0, 5)
    values[spike_idx] = 150
    spike_month = months[spike_idx]

    fig, ax = plt.subplots(figsize=(6, 3))
    colors = ["#e74c3c" if i == spike_idx else "#3498db" for i in range(6)]
    ax.bar(months, values, color=colors)
    for i, v in enumerate(values):
        ax.text(i, v + 2, str(v), ha="center", fontsize=10, fontweight="bold")
    ax.set_title(params.get("title", "Monthly Errors"))
    ax.set_ylabel("Count")

    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=80, bbox_inches="tight")
    plt.close(fig)
    img_b64 = base64.b64encode(buf.getvalue()).decode()

    return ToolResult(
        content=[
            {
                "type": "text",
                "text": f"Chart generated. Months: {months}, Values: {values}",
            },
            {
                "type": "image_url",
                "image_url": {"url": f"data:image/png;base64,{img_b64}"},
            },
        ],
        details={"spike_month": spike_month},
    )


chart_tool = Tool(
    name="generate_chart",
    description="Generate a bar chart of monthly server errors. Returns text + image.",
    parameters={
        "type": "object",
        "properties": {"title": {"type": "string", "description": "Chart title"}},
    },
    execute=chart_execute,
)

In [39]:
# Provider-aware convert_to_llm — handles OpenAI's string-only tool results
def make_convert_to_llm(model):
    is_openai = model.startswith("gpt")

    def convert(messages):
        result = []
        pending_images = []

        for m in messages:
            # Flush pending images before any non-tool message
            if pending_images and m.get("role") != "tool":
                result.append(
                    {
                        "role": "user",
                        "content": [
                            {
                                "type": "text",
                                "text": "Attached image(s) from tool result:",
                            },
                            *pending_images,
                        ],
                    }
                )
                pending_images = []

            role = m.get("role")

            if role == "assistant":
                msg = {"role": "assistant"}
                if m.get("content"):
                    msg["content"] = m["content"]
                if m.get("tool_calls"):
                    msg["tool_calls"] = m["tool_calls"]
                if m.get("thinking_blocks"):
                    msg["thinking_blocks"] = m["thinking_blocks"]
                if m.get("reasoning_content"):
                    msg["reasoning_content"] = m["reasoning_content"]
                result.append(msg)

            elif role == "user":
                result.append({"role": "user", "content": m["content"]})

            elif role == "tool":
                content = m.get("content")
                if isinstance(content, list):
                    text_parts = [b["text"] for b in content if b.get("type") == "text"]
                    images = [b for b in content if b.get("type") == "image_url"]
                    text = "\n".join(text_parts)

                    if is_openai and images:
                        # OpenAI: string-only tool results, images become synthetic user msg
                        result.append(
                            {
                                "role": "tool",
                                "tool_call_id": m["tool_call_id"],
                                "content": text or "(see attached image)",
                            }
                        )
                        pending_images.extend(images)
                    elif images:
                        # Anthropic/Gemini: content blocks with images work natively
                        result.append(
                            {
                                "role": "tool",
                                "tool_call_id": m["tool_call_id"],
                                "content": [
                                    b
                                    for b in content
                                    if b.get("type") in ("text", "image_url")
                                ],
                            }
                        )
                    else:
                        result.append(
                            {
                                "role": "tool",
                                "tool_call_id": m["tool_call_id"],
                                "content": text,
                            }
                        )
                else:
                    result.append(
                        {
                            "role": "tool",
                            "tool_call_id": m["tool_call_id"],
                            "content": content,
                        }
                    )

        # Flush remaining images
        if pending_images:
            result.append(
                {
                    "role": "user",
                    "content": [
                        {"type": "text", "text": "Attached image(s) from tool result:"},
                        *pending_images,
                    ],
                }
            )

        return result

    return convert

In [40]:
# Test all models: generate chart → ask which month has the spike
# Use logging_convert to see exactly what each provider receives
MODELS_TO_TEST = [
    "anthropic/claude-sonnet-4-6",
    "anthropic/claude-opus-4-6",
    "gemini/gemini-3-flash-preview",
    "gpt-5.2",
]

for model in MODELS_TO_TEST:
    print(f"\n{'=' * 60}")
    print(f"  {model}")
    print(f"{'=' * 60}")

    llm_calls = []
    base_convert = make_convert_to_llm(model)

    def logging_convert(messages, _base=base_convert, _calls=llm_calls):
        converted = _base(messages)
        _calls.append(converted)
        return converted

    # Turn 1: generate the chart
    context = AgentContext(
        system_prompt="You are a data analyst. Use tools when asked. Be concise.",
        messages=[],
        tools=[chart_tool],
    )
    config = AgentConfig(model=model, convert_to_llm=logging_convert)

    stream = agent_loop(
        [{"role": "user", "content": "Generate a chart of monthly server errors."}],
        context,
        config,
    )

    spike_month = None
    new_messages = []
    async for event in stream:
        if event["type"] == "tool_execution_end":
            spike_month = event["result"]["details"]["spike_month"]
            print(f"  spike month: {spike_month}")
        if (
            event["type"] == "message_end"
            and event["message"].get("role") == "assistant"
        ):
            content = event["message"].get("content")
            if content:
                print(f"  assistant: {content[:100]}...")

    new_messages = await stream.result()
    context.messages.extend(new_messages)

    # Turn 2: ask about the chart (LLM must see the image)
    llm_calls.clear()
    stream2 = agent_loop(
        [
            {
                "role": "user",
                "content": "Which month has the highest error count? Reply with just the month name.",
            }
        ],
        context,
        config,
    )

    answer = ""
    async for event in stream2:
        if (
            event["type"] == "message_end"
            and event["message"].get("role") == "assistant"
        ):
            answer = (event["message"].get("content") or "").strip()

    # Show what the LLM received on turn 2
    print("\n  --- What the LLM received (turn 2) ---")
    if llm_calls:
        for msg in llm_calls[0]:
            role = msg["role"]
            content = msg.get("content", "")
            if role == "tool":
                content_type = type(content).__name__
                if isinstance(content, list):
                    types = [b.get("type") for b in content]
                    print(f"  [{role}] content blocks: {types}")
                else:
                    print(f"  [{role}] content: {content[:60]!r}")
            elif role == "user" and isinstance(content, list):
                types = [b.get("type") for b in content]
                print(
                    f"  [{role}] content blocks: {types}  ← synthetic image injection"
                )
            else:
                c = content if isinstance(content, str) else str(content)
                print(f"  [{role}] {c[:80]!r}")

    # Did the model spot the spike?
    month_map = {
        "jan": "january",
        "feb": "february",
        "mar": "march",
        "apr": "april",
        "may": "may",
        "jun": "june",
    }
    full = month_map.get(spike_month.lower(), spike_month.lower())
    found = spike_month.lower() in answer.lower() or full in answer.lower()
    print(f"\n  answer: {answer!r}")
    print(f"  correct: {found} (expected {spike_month})")


  anthropic/claude-sonnet-4-6
  assistant: Sure! Let me generate that chart for you right away....
  spike month: Jan
  assistant: Here's the **Monthly Server Errors** bar chart! Here are some quick insights:

- 📈 **January** had a...

  --- What the LLM received (turn 2) ---
  [user] 'Generate a chart of monthly server errors.'
  [assistant] 'Sure! Let me generate that chart for you right away.'
  [tool] content blocks: ['text', 'image_url']
  [assistant] "Here's the **Monthly Server Errors** bar chart! Here are some quick insights:\n\n-"
  [user] 'Which month has the highest error count? Reply with just the month name.'

  answer: 'January'
  correct: True (expected Jan)

  anthropic/claude-opus-4-6
  assistant: 

I'll generate that chart for you right away....
  spike month: Mar
  assistant: Here's the bar chart of monthly server errors. Key observations:

- **March stands out** with a mass...

  --- What the LLM received (turn 2) ---
  [user] 'Generate a chart of monthly server er